# Sample metadata access

This notebook demonstrates the `Ag3` methods for accessing, augmenting, summarising and visualising sample-level metadata — collection details, sequence QC, cohorts, cross pedigrees, WGS data catalogs, and various tabular/map/bar-chart views over samples.

## Set up the Ag3 data resource

In [1]:
import malariagen_data
ag3 = malariagen_data.Ag3(
    "simplecache::gs://vo_agam_release_master_us_central1",
    simplecache=dict(cache_storage="../../gcs_cache"),
    results_cache="../../results_cache",
)
ag3

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


<MalariaGEN Ag3 API client>
Storage URL                           : simplecache::gs://vo_agam_release_master_us_central1
Data releases available               : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
Results cache                         : /Users/katie.barr/malariagen-data-python/results_cache
Cohorts analysis                      : 20260120
AIM analysis                          : 20220528
Site filters analysis                 : dt_20200416
Software version                      : malariagen_data 15.8.0.post13+b769b728
Client location                       : England, United Kingdom
Data filtered to unrestricted use only: False
Data filtered to surveillance use only: False
Relevant data releases                : 3.0, 3.1, 3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8, 3.9, 3.10, 3.11, 3.12, 3.13, 3.14, 3.15, 3.16
---
Please note that data are subject to terms of use,
for more information see the Vector Observatory website https://www.malariagen.net/vobs/
or contact support@malariagen.net. For API documentation see 
https://malariagen.github.io/malariagen-data-python/v15.8.0.post13+b769b728/Ag3.html

## `sample_metadata`

The core method for accessing per-sample metadata. Returns a pandas dataframe, one row per sample, assembled by merging general collection metadata, sequence QC metadata, surveillance flags, and (for Ag3) AIM species-assignment and cohort metadata — see the dataframe below for the full set of ~55 columns.

Parameters:
- `sample_sets` (optional): a single sample set or release identifier, or a sequence of them (e.g. `"3.0"`, `["AG1000G-BF-A", "AG1000G-BF-B"]`). `None` (the default) returns all relevant sample sets. Here we use `"3.0"` to keep the result to one whole release rather than the full multi-release archive.
- `sample_query` (optional): a pandas query string evaluated against the metadata to filter rows, e.g. `"country == 'Burkina Faso'"`. Below we use `"taxon == 'coluzzii' and year >= 2012"` to combine a categorical and a numeric filter, showing how conditions compose. Mutually exclusive with `sample_indices`.
- `sample_query_options` (optional): a dict of extra kwargs passed through to pandas' `query()`/`eval()` (e.g. `parser`, `engine`), for advanced query needs.
- `sample_indices` (optional): a list of integer positions (aligned to the order in the underlying metadata) to select samples directly, as an alternative to `sample_query`.

Note: lab-cross samples use `year=-1`/`month=-1` as sentinels for "no real collection date"; our query's `year >= 2012` naturally excludes those.

In [2]:
df_samples = ag3.sample_metadata(
    sample_sets="3.0", sample_query="taxon == 'coluzzii' and year >= 2012"
)
df_samples

Load sample metadata: ⠋ (0:00:00.00)

Load sample metadata: ⠙ (0:00:00.11)

Load sample metadata: ⠹ (0:00:00.19)

Load sample metadata: ⠸ (0:00:00.28)

Load sample metadata: ⠼ (0:00:00.37)

Load sample metadata: ⠴ (0:00:00.46)

,sample_id,partner_sample_id,contributor,country,location,year,month,latitude,longitude,sex_call,...,admin1_name,admin1_iso,admin2_name,taxon,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,AB0087-C,BF3-3,Austin Burt,Burkina Faso,Bana,2012,7,11.233,-4.472,F,...,Hauts-Bassins,BF-09,Houet,coluzzii,BF-09_colu_2012,BF-09_colu_2012_07,BF-09_colu_2012_Q3,BF-09_Houet_colu_2012,BF-09_Houet_colu_2012_07,BF-09_Houet_colu_2012_Q3
1,AB0088-C,BF3-5,Austin Burt,Burkina Faso,Bana,2012,7,11.233,-4.472,F,...,Hauts-Bassins,BF-09,Houet,coluzzii,BF-09_colu_2012,BF-09_colu_2012_07,BF-09_colu_2012_Q3,BF-09_Houet_colu_2012,BF-09_Houet_colu_2012_07,BF-09_Houet_colu_2012_Q3
2,AB0089-Cx,BF3-8,Austin Burt,Burkina Faso,Bana,2012,7,11.233,-4.472,F,...,Hauts-Bassins,BF-09,Houet,coluzzii,BF-09_colu_2012,BF-09_colu_2012_07,BF-09_colu_2012_Q3,BF-09_Houet_colu_2012,BF-09_Houet_colu_2012_07,BF-09_Houet_colu_2012_Q3
3,AB0090-C,BF3-10,Austin Burt,Burkina Faso,Bana,2012,7,11.233,-4.472,F,...,Hauts-Bassins,BF-09,Houet,coluzzii,BF-09_colu_2012,BF-09_colu_2012_07,BF-09_colu_2012_Q3,BF-09_Houet_colu_2012,BF-09_Houet_colu_2012_07,BF-09_Houet_colu_2012_Q3
4,AB0091-C,BF3-12,Austin Burt,Burkina Faso,Bana,2012,7,11.233,-4.472,F,...,Hauts-Bassins,BF-09,Houet,coluzzii,BF-09_colu_2012,BF-09_colu_2012_07,BF-09_colu_2012_Q3,BF-09_Houet_colu_2012,BF-09_Houet_colu_2012_07,BF-09_Houet_colu_2012_Q3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
530,AZ0433-CW,MA11-13,Austin Burt,Mali,Tieneguebougou,2014,8,12.810,-8.080,F,...,Koulikouro,ML-2,Kati,coluzzii,ML-2_colu_2014,ML-2_colu_2014_08,ML-2_colu_2014_Q3,ML-2_Kati_colu_2014,ML-2_Kati_colu_2014_08,ML-2_Kati_colu_2014_Q3
531,AZ0434-CW,MA11-14,Austin Burt,Mali,Tieneguebougou,2014,8,12.810,-8.080,F,...,Koulikouro,ML-2,Kati,coluzzii,ML-2_colu_2014,ML-2_colu_2014_08,ML-2_colu_2014_Q3,ML-2_Kati_colu_2014,ML-2_Kati_colu_2014_08,ML-2_Kati_colu_2014_Q3
532,AZ0416-CW,MA10-32,Austin Burt,Mali,Tieneguebougou,2014,8,12.810,-8.080,F,...,Koulikouro,ML-2,Kati,coluzzii,ML-2_colu_2014,ML-2_colu_2014_08,ML-2_colu_2014_Q3,ML-2_Kati_colu_2014,ML-2_Kati_colu_2014_08,ML-2_Kati_colu_2014_Q3
533,AZ0444-CW,MA11-28,Austin Burt,Mali,Tieneguebougou,2014,8,12.810,-8.080,F,...,Koulikouro,ML-2,Kati,coluzzii,ML-2_colu_2014,ML-2_colu_2014_08,ML-2_colu_2014_Q3,ML-2_Kati_colu_2014,ML-2_Kati_colu_2014_08,ML-2_Kati_colu_2014_Q3


## `add_extra_metadata`

Attaches a user-supplied dataframe of extra per-sample columns, which then get merged into subsequent `sample_metadata()` calls (and everything built on top of it, e.g. `sample_query`). Useful for bringing in your own annotations (phenotypes, custom groupings, etc.) without editing the underlying data.

Parameters:
- `data` (required): a dataframe with one row per sample, containing either a `sample_id` or `partner_sample_id` column (values must be unique) plus whatever extra columns you want to add.
- `on` (default `"sample_id"`): which of `data`'s columns to merge on — must be `"sample_id"` or `"partner_sample_id"`.

Below we tag each coluzzii sample from the query above with a made-up `collection_note`, then show that the extra column appears in `sample_metadata()`.

In [3]:
import numpy as np
import pandas as pd

df_extra = pd.DataFrame(
    {
        "sample_id": df_samples["sample_id"],
        "collection_note": np.random.choice(
            ["priority", "routine", "followup"], size=len(df_samples)
        ),
    }
)
ag3.add_extra_metadata(df_extra, on="sample_id")
ag3.sample_metadata(sample_sets="3.0", sample_query="taxon == 'coluzzii' and year >= 2012")[
    ["sample_id", "country", "year", "collection_note"]
]

Load sample metadata: ⠋ (0:00:00.00)

Load sample metadata: ⠙ (0:00:00.11)

Load sample metadata: ⠹ (0:00:00.20)

Load sample metadata: ⠸ (0:00:00.29)

Load sample metadata: ⠼ (0:00:00.39)

...

Load sample metadata: ⠦ (0:00:01.56)

Load sample metadata: ⠧ (0:00:01.65)

Load sample metadata: ⠇ (0:00:01.73)

Load sample metadata: ⠏ (0:00:01.82)

,sample_id,country,year,collection_note
0,AB0087-C,Burkina Faso,2012,priority
1,AB0088-C,Burkina Faso,2012,priority
2,AB0089-Cx,Burkina Faso,2012,priority
3,AB0090-C,Burkina Faso,2012,routine
4,AB0091-C,Burkina Faso,2012,priority
...,...,...,...,...
530,AZ0433-CW,Mali,2014,routine
531,AZ0434-CW,Mali,2014,routine
532,AZ0416-CW,Mali,2014,priority
533,AZ0444-CW,Mali,2014,followup


## `clear_extra_metadata`

Removes all extra metadata previously attached via `add_extra_metadata`, reverting `sample_metadata()` to only the built-in columns. Takes no parameters. Useful to reset state between experiments in a long-running session/notebook.

In [4]:
ag3.clear_extra_metadata()
"collection_note" in ag3.sample_metadata(sample_sets="3.0").columns

False

## `cross_metadata`

Loads metadata for the laboratory colony cross samples (the `AG1000G-X` sample set): which samples are parents vs. progeny in each cross, and their family relationships. Returns a dataframe with columns `cross`, `sample_id`, `father_id`, `mother_id`, `sex`, and `role` (`"parent"` or `"progeny"`). Takes no parameters.

In [5]:
ag3.cross_metadata()

,cross,sample_id,father_id,mother_id,sex,role
0,18-5,AD0142-C,NaN,NaN,F,parent
1,18-5,AD0143-C,NaN,NaN,M,parent
2,18-5,AD0163-C,AD0143-C,AD0142-C,M,progeny
3,18-5,AD0162-C,AD0143-C,AD0142-C,F,progeny
4,18-5,AD0161-C,AD0143-C,AD0142-C,M,progeny
...,...,...,...,...,...,...
293,K6,AC0346-C,AC0398-C,AC0334-C,M,progeny
294,K6,AC0347-C,AC0398-C,AC0334-C,M,progeny
295,K6,AC0348-C,AC0398-C,AC0334-C,M,progeny
296,K6,AC0335-C,AC0398-C,AC0334-C,M,progeny


## `count_samples`

Builds a pivot table of sample counts by space/time/taxon. Rows are a spatio-temporal index, columns are (by default) taxon.

Parameters:
- `sample_sets`, `sample_query`, `sample_query_options`, `sample_indices`: the same sample-selection parameters as `sample_metadata`, applied before counting.
- `index` (default `("country", "admin1_iso", "admin1_name", "admin2_name", "year")`): which metadata columns form the pivot table's row index.
- `columns` (default `"taxon"`): which metadata column(s) to spread across the pivot table's columns.

Below we narrow the row index to `("country", "year")` and pivot on `"sex_call"` instead of the default `"taxon"`, to show a more compact, differently-shaped table than the default.

In [6]:
ag3.count_samples(sample_sets="3.0", index=["country", "year"], columns="sex_call")

sex_call                                  F    M  UKN
country                          year                
Angola                            2009   77    4    0
Burkina Faso                      2004   13    0    0
                                  2012  151   30    0
                                  2014   74   28    0
Cameroon                          2005   97    0    0
                                  2009  258   45    0
                                  2013   44    0    0
Central African Republic          1993    7    0    0
                                  1994   66    0    0
Cote d'Ivoire                     2012   80    0    0
Democratic Republic of the Congo  2015   44   32    0
Equatorial Guinea                 2002   10    0    0
Gabon                             2000   69    0    0
Gambia, The                       2006    2    0   29
                                  2011   74    0    0
                                  2012  174    0    0
Ghana                             2012  100    0    0
Guinea                            2012  112   24    0
Guinea-Bissau                     2010  101    0    0
Kenya                             2000    8   11    0
                                  2007    3    0    0
                                  2012   64    0    0
Lab Cross                        -1     132  165    0
Malawi                            2015   41    0    0
Mali                              2004   71    0    0
                                  2012   77   17    0
                                  2014   51    9    0
Mayotte                           2011   11   12    0
Mozambique                        2003    3    0    0
                                  2004   71    0    0
Tanzania                          2012   86    1    0
                                  2013   43    0    0
                                  2015  160   10    0
Uganda                            2012  290    0    0

## `lookup_sample`

Returns the full metadata record (as a pandas `Series`) for one specific sample.

Parameters:
- `sample` (required): either the sample's string `sample_id`, or its integer positional index within the resolved metadata.
- `sample_set` (optional): restrict the lookup to a specific sample set; if `None` (default), metadata for all relevant sample sets is loaded and searched, which is slower but more flexible.

In [7]:
ag3.lookup_sample(sample="AB0087-C", sample_set="AG1000G-BF-A")

Load sample metadata: ⠋ (0:00:00.00)

partner_sample_id                                                              BF3-3
contributor                                                              Austin Burt
country                                                                 Burkina Faso
location                                                                        Bana
year                                                                            2012
month                                                                              7
latitude                                                                      11.233
longitude                                                                     -4.472
sex_call                                                                           F
sample_set                                                              AG1000G-BF-A
release                                                                          3.0
quarter                                                          

## `wgs_run_accessions`

Loads a table mapping samples in a given sample set to their ENA (European Nucleotide Archive) sequencing run accessions. Returns a dataframe with columns `sample_id` and `run_ena`.

Parameters:
- `sample_set` (required): the sample set to load accessions for, e.g. `"AG1000G-BF-A"`.

In [8]:
ag3.wgs_run_accessions(sample_set="AG1000G-BF-A")

,sample_id,run_ena
0,AB0085-Cx,"ERR495554, ERR501731, ERR501743"
1,AB0086-Cx,"ERR491321, ERR495433, ERR495445"
2,AB0087-C,"ERR328834, ERR332017, ERR332029"
3,AB0088-C,"ERR328830, ERR332001, ERR332013"
4,AB0089-Cx,"ERR495787, ERR502048, ERR502132"
...,...,...
176,AB0280-Cx,"ERR495808, ERR502057, ERR502153"
177,AB0281-Cx,"ERR491202, ERR495686, ERR501923"
178,AB0282-Cx,"ERR491337, ERR495473, ERR495485"
179,AB0283-C,"ERR323754, ERR327004, ERR327016"


## `plot_samples_bar`

Plots a plotly bar chart of sample counts grouped by a chosen metadata variable (and optionally a second, for stacking/colouring).

Parameters:
- `x` (required): metadata column for the X axis / grouping, e.g. `"country"` or `"year"` (samples with missing `year` are dropped automatically when `x="year"`).
- `sample_sets`, `sample_query`, `sample_query_options`, `sample_indices`: sample-selection parameters as above.
- `color` (optional): a second metadata column to colour/stack bars by, e.g. `"taxon"`.
- `sort` (default `True`): if `True`, order the bars by count (ascending); if `False`, use the natural/categorical order of the `x` values.
- `template` (default `"plotly_white"`): plotly figure theme.
- `width`, `height` (default `800`, `600`): figure size in pixels.
- `show` (default `True`): display immediately vs. return the figure object.
- `renderer` (optional): plotly renderer name to use when displaying (e.g. `"notebook"`, `"png"`); `None` uses plotly's default.
- `**kwargs`: passed through to `plotly.express.bar()` for further customisation.

Below we plot sample counts by `country`, coloured by `taxon`, sorted by size.

In [9]:
ag3.plot_samples_bar(x="country", color="taxon", sample_sets="3.0", sort=True)

## `plot_samples_interactive_map`

Builds an interactive `ipyleaflet` map with one marker per distinct sampling location, where hovering shows a popup summarising years, sample sets, contributors and per-taxon sample counts at that location.

Parameters:
- `sample_sets`, `sample_query`, `sample_query_options`, `sample_indices`: sample-selection parameters.
- `basemap` (optional): an ipyleaflet basemap, or an abbreviation string (e.g. `"mapnik"`, `"satellite"`, `"positron"`); `None` (default) uses `Esri.WorldImagery`.
- `center` (default `(-2, 20)`): initial map centre as `(latitude, longitude)`.
- `zoom` (default `3`): initial zoom level.
- `height`, `width` (defaults from `map_params`): map widget size in pixels (or CSS size strings).
- `min_samples` (default `1`): only show a marker for a location if it has at least this many samples in total, useful for hiding singleton locations on a crowded map.
- `count_by` (default `"taxon"`): metadata column used to break down the per-location sample counts shown in each marker's popup.

Below we restrict to Burkina Faso samples, use the `"positron"` basemap abbreviation, and require at least 5 samples per marker.

In [10]:
ag3.plot_samples_interactive_map(
    sample_sets="3.0",
    sample_query="country == 'Burkina Faso'",
    basemap="positron",
    min_samples=5,
)

/opt/homebrew/Caskroom/miniconda/base/envs/malariagen2/lib/python3.11/site-packages/ipyleaflet/leaflet.py:105: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  url = basemap.build_url(time=day, scale_factor="{r}")


Map(center=[-2, 20], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

## `plot_sample_location_mapbox`

Plots sample collection locations as an interactive plotly map-tile scatter plot (via `px.scatter_map`, using OpenStreetMap tiles), one point per distinct location, coloured by a chosen metadata variable.

Parameters:
- `sample_sets` (required, keyword-only): sample-selection as above.
- `sample_query`, `sample_query_options`: further sample filtering.
- `marker_size` (default `10`): size of the location markers.
- `color` (default `"admin1_name"`): metadata column used to colour points (and to sort the deduplicated location table before plotting). Note this is applied to a table of *distinct locations*, so it must be one of the location-identifying columns (`country`, `admin1_iso`, `admin1_name`, `admin2_name`, `location`, `latitude`, `longitude`) — a per-sample column like `taxon` will raise a `KeyError` since it isn't present once locations are deduplicated.
- `color_discrete_sequence` (default: the Plotly "Prism" qualitative palette): explicit list of colours to cycle through for `color` categories.
- `category_orders` (optional): control the order categories appear in, e.g. in the legend.
- `hover_name` (default `"location"`): column shown in bold in the hover tooltip.
- `zoom` (optional): initial map zoom; `None` lets plotly auto-fit.
- `width`, `height` (default `800`, `600`): figure size in pixels.
- `show` (default `True`): display immediately vs. return the figure.
- `renderer` (optional): plotly renderer name.
- `**kwargs`: passed through to `px.scatter_map()`.

Below we plot Burkina Faso locations, coloured by `admin2_name` instead of the default `admin1_name`.

In [11]:
ag3.plot_sample_location_mapbox(
    sample_sets="3.0",
    sample_query="country == 'Burkina Faso'",
    color="admin2_name",
)

## `plot_sample_location_geo`

Plots sample collection locations as a plotly geographic scatter plot (`px.scatter_geo`, vector map rather than map tiles) — better suited to continental/global overviews than `plot_sample_location_mapbox`.

Parameters:
- `sample_sets` (required, keyword-only), `sample_query`, `sample_query_options`, `sample_indices`: sample-selection parameters.
- `marker_size` (default `10`): marker size.
- `color` (default `"admin1_name"`): metadata column used to colour points.
- `color_discrete_sequence` (default: Plotly "Prism" palette): colours for `color` categories.
- `category_orders` (optional): category ordering.
- `hover_name` (default `"location"`): bold tooltip field.
- `fitbounds` (default `"locations"`): how the view auto-fits to the data (`"locations"` fits to visible points; `False` disables auto-fit; `"geojson"` fits to supplied geometry).
- `scope` (default `"world"`): map scope/continent to draw, e.g. `"africa"`, `"world"`.
- `width`, `height`, `show`, `renderer`, `**kwargs`: as in `plot_sample_location_mapbox`.

Below we set `scope="africa"` (a more illustrative choice than the world default, given all Ag3 samples come from Africa) and colour by `country`.

In [12]:
ag3.plot_sample_location_geo(
    sample_sets="3.0",
    color="country",
    scope="africa",
)

## `wgs_data_catalog`

Loads a data catalog of download URLs (BAM alignments, SNP-genotype VCF, SNP-genotype Zarr) for every sample in a given sample set.

Parameters:
- `sample_set` (required): the sample set to load the catalog for, e.g. `"AG1000G-BF-A"`.

In [13]:
ag3.wgs_data_catalog(sample_set="AG1000G-BF-A")

,sample_id,alignments_bam,snp_genotypes_vcf,snp_genotypes_zarr
0,AB0085-Cx,https://vo_agam_output.cog.sanger.ac.uk/AB0085...,https://vo_agam_output.cog.sanger.ac.uk/AB0085...,https://vo_agam_output.cog.sanger.ac.uk/AB0085...
1,AB0086-Cx,https://vo_agam_output.cog.sanger.ac.uk/AB0086...,https://vo_agam_output.cog.sanger.ac.uk/AB0086...,https://vo_agam_output.cog.sanger.ac.uk/AB0086...
2,AB0087-C,https://vo_agam_output.cog.sanger.ac.uk/AB0087...,https://vo_agam_output.cog.sanger.ac.uk/AB0087...,https://vo_agam_output.cog.sanger.ac.uk/AB0087...
3,AB0088-C,https://vo_agam_output.cog.sanger.ac.uk/AB0088...,https://vo_agam_output.cog.sanger.ac.uk/AB0088...,https://vo_agam_output.cog.sanger.ac.uk/AB0088...
4,AB0089-Cx,https://vo_agam_output.cog.sanger.ac.uk/AB0089...,https://vo_agam_output.cog.sanger.ac.uk/AB0089...,https://vo_agam_output.cog.sanger.ac.uk/AB0089...
...,...,...,...,...
176,AB0280-Cx,https://vo_agam_output.cog.sanger.ac.uk/AB0280...,https://vo_agam_output.cog.sanger.ac.uk/AB0280...,https://vo_agam_output.cog.sanger.ac.uk/AB0280...
177,AB0281-Cx,https://vo_agam_output.cog.sanger.ac.uk/AB0281...,https://vo_agam_output.cog.sanger.ac.uk/AB0281...,https://vo_agam_output.cog.sanger.ac.uk/AB0281...
178,AB0282-Cx,https://vo_agam_output.cog.sanger.ac.uk/AB0282...,https://vo_agam_output.cog.sanger.ac.uk/AB0282...,https://vo_agam_output.cog.sanger.ac.uk/AB0282...
179,AB0283-C,https://vo_agam_output.cog.sanger.ac.uk/AB0283...,https://vo_agam_output.cog.sanger.ac.uk/AB0283...,https://vo_agam_output.cog.sanger.ac.uk/AB0283...


## `cohorts`

Loads a table describing pre-defined sample cohorts for a given cohort set — spatio-temporal groupings (e.g. by admin-level region and year/quarter/month) with derived summary columns such as representative lat/lon and country ISO codes.

Parameters:
- `cohort_set` (required): one of `"admin1_month"`, `"admin1_quarter"`, `"admin1_year"`, `"admin2_month"`, `"admin2_quarter"`, `"admin2_year"` — sets the spatial level (admin1 vs. admin2) and temporal resolution (month/quarter/year) used to define cohorts.
- `query` (optional): a pandas query string to filter the returned cohorts table, e.g. by country or minimum sample count.

Below we use `cohort_set="admin1_year"` and filter to Burkina Faso cohorts with `query`.

In [14]:
ag3.cohorts(cohort_set="admin1_year", query="country == 'Burkina Faso'")

,cohort_id,cohort_size,country,country_alpha2,country_alpha3,taxon,year,admin1_name,admin1_iso,admin1_geoboundaries_shape_id,admin1_representative_longitude,admin1_representative_latitude
0,BF-01_arab_2008,1,Burkina Faso,BF,BFA,arabiensis,2008,Boucle du Mouhoun,BF-01,92566538B98190668782446,-3.592255,12.479899
1,BF-01_arab_2022,13,Burkina Faso,BF,BFA,arabiensis,2022,Boucle du Mouhoun,BF-01,92566538B98190668782446,-3.592255,12.479899
2,BF-01_colu_2008,4,Burkina Faso,BF,BFA,coluzzii,2008,Boucle du Mouhoun,BF-01,92566538B98190668782446,-3.592255,12.479899
3,BF-01_colu_2022,32,Burkina Faso,BF,BFA,coluzzii,2022,Boucle du Mouhoun,BF-01,92566538B98190668782446,-3.592255,12.479899
4,BF-01_gamb_2022,2,Burkina Faso,BF,BFA,gambiae,2022,Boucle du Mouhoun,BF-01,92566538B98190668782446,-3.592255,12.479899
5,BF-02_arab_2022,8,Burkina Faso,BF,BFA,arabiensis,2022,Cascades,BF-02,92566538B44525923588019,-4.482810,10.308460
6,BF-02_colu_2011,18,Burkina Faso,BF,BFA,coluzzii,2011,Cascades,BF-02,92566538B44525923588019,-4.482810,10.308460
7,BF-02_colu_2012,63,Burkina Faso,BF,BFA,coluzzii,2012,Cascades,BF-02,92566538B44525923588019,-4.482810,10.308460
8,BF-02_colu_2015,33,Burkina Faso,BF,BFA,coluzzii,2015,Cascades,BF-02,92566538B44525923588019,-4.482810,10.308460
9,BF-02_colu_2016,53,Burkina Faso,BF,BFA,coluzzii,2016,Cascades,BF-02,92566538B44525923588019,-4.482810,10.308460


## `cohorts_metadata`

Loads per-sample cohort membership metadata (which cohort each sample falls into, under the currently-configured cohorts analysis), separately from the combined `sample_metadata()`.

Parameters:
- `sample_sets` (optional): sample-selection as above; `None` returns all relevant sample sets.

In [15]:
ag3.cohorts_metadata(sample_sets="3.0")

,sample_id,country_iso,admin1_name,admin1_iso,admin2_name,taxon,cohort_admin1_year,cohort_admin1_month,cohort_admin1_quarter,cohort_admin2_year,cohort_admin2_month,cohort_admin2_quarter
0,AR0047-C,AGO,Luanda,AO-LUA,Luanda,coluzzii,AO-LUA_colu_2009,AO-LUA_colu_2009_04,AO-LUA_colu_2009_Q2,AO-LUA_Luanda_colu_2009,AO-LUA_Luanda_colu_2009_04,AO-LUA_Luanda_colu_2009_Q2
1,AR0049-C,AGO,Luanda,AO-LUA,Luanda,coluzzii,AO-LUA_colu_2009,AO-LUA_colu_2009_04,AO-LUA_colu_2009_Q2,AO-LUA_Luanda_colu_2009,AO-LUA_Luanda_colu_2009_04,AO-LUA_Luanda_colu_2009_Q2
2,AR0051-C,AGO,Luanda,AO-LUA,Luanda,coluzzii,AO-LUA_colu_2009,AO-LUA_colu_2009_04,AO-LUA_colu_2009_Q2,AO-LUA_Luanda_colu_2009,AO-LUA_Luanda_colu_2009_04,AO-LUA_Luanda_colu_2009_Q2
3,AR0061-C,AGO,Luanda,AO-LUA,Luanda,coluzzii,AO-LUA_colu_2009,AO-LUA_colu_2009_04,AO-LUA_colu_2009_Q2,AO-LUA_Luanda_colu_2009,AO-LUA_Luanda_colu_2009_04,AO-LUA_Luanda_colu_2009_Q2
4,AR0078-C,AGO,Luanda,AO-LUA,Luanda,coluzzii,AO-LUA_colu_2009,AO-LUA_colu_2009_04,AO-LUA_colu_2009_Q2,AO-LUA_Luanda_colu_2009,AO-LUA_Luanda_colu_2009_04,AO-LUA_Luanda_colu_2009_Q2
...,...,...,...,...,...,...,...,...,...,...,...,...
3076,AD0494-C,NaN,NaN,NaN,NaN,unassigned,NaN,NaN,NaN,NaN,NaN,NaN
3077,AD0495-C,NaN,NaN,NaN,NaN,unassigned,NaN,NaN,NaN,NaN,NaN,NaN
3078,AD0496-C,NaN,NaN,NaN,NaN,unassigned,NaN,NaN,NaN,NaN,NaN,NaN
3079,AD0497-C,NaN,NaN,NaN,NaN,unassigned,NaN,NaN,NaN,NaN,NaN,NaN


## `cohort_geometries`

Loads the GeoJSON boundary geometries (a `FeatureCollection` dict) corresponding to the cohorts in a given cohort set — i.e. the polygon/administrative-boundary shapes to draw on a map alongside `cohorts()` data.

Parameters:
- `cohort_set` (required): same six accepted values as for `cohorts()` (e.g. `"admin1_year"`); must match the cohort set you intend to overlay this geometry with.

In [16]:
geojson = ag3.cohort_geometries(cohort_set="admin1_year")
type(geojson), geojson["type"], len(geojson["features"]), geojson["features"][0]["properties"]

(dict, 'FeatureCollection', 547, {'cohort_id': 'AO-LUA_colu_2009'})

## `general_metadata`

Loads only the "general" (collection-level) sample metadata component — sample IDs, contributor, location, date, sex call, plus study/terms-of-use columns — without the sequence-QC, surveillance, AIM or cohort components that `sample_metadata()` merges in. Faster/lighter when you only need basic collection info.

Parameters:
- `sample_sets` (optional): sample-selection as above; `None` returns all relevant sample sets.

In [17]:
ag3.general_metadata(sample_sets="AG1000G-BF-A")

,sample_id,partner_sample_id,contributor,country,location,year,month,latitude,longitude,sex_call,sample_set,release,quarter,study_id,study_url,terms_of_use_expiry_date,terms_of_use_url,unrestricted_use
0,AB0085-Cx,BF2-4,Austin Burt,Burkina Faso,Pala,2012,7,11.151,-4.235,F,AG1000G-BF-A,3.0,3,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,True
1,AB0086-Cx,BF2-6,Austin Burt,Burkina Faso,Pala,2012,7,11.151,-4.235,F,AG1000G-BF-A,3.0,3,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,True
2,AB0087-C,BF3-3,Austin Burt,Burkina Faso,Bana,2012,7,11.233,-4.472,F,AG1000G-BF-A,3.0,3,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,True
3,AB0088-C,BF3-5,Austin Burt,Burkina Faso,Bana,2012,7,11.233,-4.472,F,AG1000G-BF-A,3.0,3,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,True
4,AB0089-Cx,BF3-8,Austin Burt,Burkina Faso,Bana,2012,7,11.233,-4.472,F,AG1000G-BF-A,3.0,3,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,AB0280-Cx,BF12-31,Austin Burt,Burkina Faso,Pala,2012,7,11.151,-4.235,F,AG1000G-BF-A,3.0,3,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,True
177,AB0281-Cx,BF12-32,Austin Burt,Burkina Faso,Pala,2012,7,11.151,-4.235,F,AG1000G-BF-A,3.0,3,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,True
178,AB0282-Cx,BF12-33,Austin Burt,Burkina Faso,Pala,2012,7,11.151,-4.235,F,AG1000G-BF-A,3.0,3,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,True
179,AB0283-C,BF10-12,Austin Burt,Burkina Faso,Pala,2012,7,11.151,-4.235,F,AG1000G-BF-A,3.0,3,AG1000G-BF-1,https://www.malariagen.net/partner_study/AG100...,2025-01-01,https://www.malariagen.net/data/our-approach-s...,True


## `sequence_qc_metadata`

Loads only the sequence quality-control metadata component (coverage statistics genome-wide and per contig, genome coverage fraction, divergence, contamination estimates) for one or more sample sets, without the other components `sample_metadata()` merges in.

Parameters:
- `sample_sets` (optional): sample-selection as above; `None` returns all relevant sample sets.

In [18]:
ag3.sequence_qc_metadata(sample_sets="AG1000G-BF-A")

,sample_id,mean_cov,median_cov,modal_cov,mean_cov_2L,median_cov_2L,mode_cov_2L,mean_cov_2R,median_cov_2R,mode_cov_2R,...,mean_cov_3R,median_cov_3R,mode_cov_3R,mean_cov_X,median_cov_X,mode_cov_X,frac_gen_cov,divergence,contam_pct,contam_LLR
0,AB0085-Cx,30.62,31,31,30.72,31,31,30.43,31,31,...,30.47,31,31,31.4,30,30,0.935,0.022,1.193,1503.154
1,AB0086-Cx,29.83,30,30,29.88,30,30,29.6,30,30,...,29.77,30,30,30.35,29,29,0.938,0.022,3.917,8025.882
2,AB0087-C,38.23,38,38,37.82,37,37,37.85,38,38,...,38.3,38,38,40.02,38,36,0.941,0.02,0.655,682.111
3,AB0088-C,24.32,24,24,24.2,24,24,24.11,24,24,...,24.14,24,24,25.9,24,24,0.939,0.02,1.023,829.1
4,AB0089-Cx,16.93,17,17,16.78,16,16,16.6,16,16,...,16.93,17,17,18.12,17,17,0.934,0.02,0.414,140.574
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,AB0280-Cx,30.92,31,31,31.18,31,31,30.93,31,32,...,30.48,31,31,31.64,31,31,0.934,0.022,0.451,271.13
177,AB0281-Cx,22.1,22,21,22.04,22,21,21.86,22,22,...,22.11,22,21,23.26,22,21,0.922,0.022,0.227,62.509
178,AB0282-Cx,27.15,27,27,27.14,27,27,27.02,27,27,...,26.91,27,27,28.47,27,27,0.938,0.02,0.681,513.851
179,AB0283-C,30.11,30,31,30.44,31,31,29.92,30,32,...,30.23,30,31,29.63,29,30,0.935,0.022,0.84,800.897
